In [ ]:
from decimal import Decimal, ROUND_HALF_UP


def validate_heatmap_inputs(results, metrics, model_order):
    required_columns = {"threshold", "model", *metrics}
    missing_columns = sorted(required_columns - set(results.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    if results["model"].nunique() != 2:
        raise ValueError("This heatmap requires exactly two models.")

    if model_order is not None and set(model_order) != set(results["model"].unique()):
        raise ValueError("model_order must match the two model names in results.")

    duplicate_rows = results.duplicated(subset=["threshold", "model"]).any()
    if duplicate_rows:
        raise ValueError("Each threshold-model combination must appear only once.")


def resolve_model_order(results, model_order=None):
    if model_order is not None:
        return model_order
    return sorted(results["model"].unique().tolist())


def build_metric_table(results, metric, model_order):
    metric_table = results.pivot(
        index="threshold",
        columns="model",
        values=metric,
    )
    return metric_table[model_order].sort_index()


def quantize_metric_value(value, decimals=3):
    if pd.isna(value):
        return None

    quantizer = Decimal("1").scaleb(-decimals)
    return Decimal(str(float(value))).quantize(quantizer, rounding=ROUND_HALF_UP)


def format_metric_value(value, decimals=3):
    quantized_value = quantize_metric_value(value, decimals=decimals)
    if quantized_value is None:
        return "NA"
    return format(quantized_value, f".{decimals}f")


def choose_winner_from_values(
    value_a,
    value_b,
    model_a,
    model_b,
    higher_is_better,
    tie_decimals=3,
):
    if pd.isna(value_a) and pd.isna(value_b):
        return "Tie"
    if pd.isna(value_a):
        return model_b
    if pd.isna(value_b):
        return model_a
    rounded_a = quantize_metric_value(value_a, decimals=tie_decimals)
    rounded_b = quantize_metric_value(value_b, decimals=tie_decimals)
    if rounded_a == rounded_b:
        return "Tie"

    if higher_is_better:
        return model_a if rounded_a > rounded_b else model_b
    return model_a if rounded_a < rounded_b else model_b


def build_winner_table(results, metrics, metric_directions, model_order, tie_decimals=3):
    winner_columns = {}

    for metric in metrics:
        metric_table = build_metric_table(results, metric, model_order)
        model_a, model_b = model_order

        winners = metric_table.apply(
            lambda row: choose_winner_from_values(
                value_a=row[model_a],
                value_b=row[model_b],
                model_a=model_a,
                model_b=model_b,
                higher_is_better=metric_directions[metric],
                tie_decimals=tie_decimals,
            ),
            axis=1,
        )
        winner_columns[metric] = winners

    winner_table = pd.DataFrame(winner_columns)
    winner_table.index.name = "threshold"
    return winner_table


def build_annotation_table(results, metrics, model_order, model_labels=None, annotation_decimals=3):
    model_labels = model_labels or {}
    annotations = {}

    for metric in metrics:
        metric_table = build_metric_table(results, metric, model_order)
        model_a, model_b = model_order
        label_a = model_labels.get(model_a, model_a)
        label_b = model_labels.get(model_b, model_b)

        annotations[metric] = metric_table.apply(
            lambda row: (
                f"{label_a}: {format_metric_value(row[model_a], decimals=annotation_decimals)}\n"
                f"{label_b}: {format_metric_value(row[model_b], decimals=annotation_decimals)}"
            ),
            axis=1,
        )

    annotation_table = pd.DataFrame(annotations)
    annotation_table.index.name = "threshold"
    return annotation_table


def prettify_metric_labels(metrics):
    return [metric.replace("_", " ").title() for metric in metrics]


def plot_threshold_winner_heatmap(
    results,
    metrics,
    metric_directions,
    title="Which Model Performs Better Across Decision Thresholds?",
    model_order=None,
    model_labels=None,
    metric_labels=None,
    output_path=None,
    figsize=(12, 7),
    tie_decimals=3,
):
    validate_heatmap_inputs(results, metrics, model_order)
    model_order = resolve_model_order(results, model_order)
    model_a, model_b = model_order

    winner_table = build_winner_table(
        results=results,
        metrics=metrics,
        metric_directions=metric_directions,
        model_order=model_order,
        tie_decimals=tie_decimals,
    )

    annotation_table = build_annotation_table(
        results=results,
        metrics=metrics,
        model_order=model_order,
        model_labels=model_labels,
        annotation_decimals=tie_decimals,
    )

    code_map = {model_a: 0, model_b: 1, "Tie": 2}
    encoded = winner_table.apply(lambda col: col.map(code_map)).astype(int)

    cmap = ListedColormap(["#4C78A8", "#F58518", "#BDBDBD"])
    norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], cmap.N)

    sns.set_theme(style="white", font_scale=0.95)
    fig, ax = plt.subplots(figsize=figsize)

    sns.heatmap(
        encoded,
        annot=annotation_table,
        fmt="",
        cmap=cmap,
        norm=norm,
        cbar=False,
        linewidths=1,
        linecolor="white",
        annot_kws={"fontsize": 9, "fontweight": "semibold"},
        ax=ax,
    )

    display_labels = model_labels or {}
    x_labels = metric_labels if metric_labels is not None else prettify_metric_labels(metrics)
    y_labels = [f"{float(threshold):.2%}" for threshold in encoded.index]

    ax.set_title(title, fontsize=14, fontweight="bold", pad=14)
    ax.set_xlabel("Metric", fontsize=11)
    ax.set_ylabel("Decision Threshold", fontsize=11)
    ax.set_xticklabels(x_labels, rotation=0, ha="center")
    ax.set_yticklabels(y_labels, rotation=0)

    ax.legend(
        handles=[
            mpatches.Patch(color="#4C78A8", label=f"{display_labels.get(model_a, model_a)} wins"),
            mpatches.Patch(color="#F58518", label=f"{display_labels.get(model_b, model_b)} wins"),
            mpatches.Patch(color="#BDBDBD", label="Tie"),
        ],
        title="Better Model",
        loc="upper center",
        bbox_to_anchor=(0.5, -0.12),
        ncol=3,
        frameon=False,
    )

    fig.tight_layout()

    if output_path is not None:
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    return winner_table, annotation_table, fig


def build_type_error_rate_plot_data(results):
    plot_data = (
        results[
            ["model", "threshold", "type_I_error_rate", "type_II_error_rate"]
        ]
        .melt(
            id_vars=["model", "threshold"],
            value_vars=["type_I_error_rate", "type_II_error_rate"],
            var_name="error_type",
            value_name="error_rate",
        )
    )

    plot_data["error_type"] = plot_data["error_type"].map({
        "type_I_error_rate": "Type I Error",
        "type_II_error_rate": "Type II Error",
    })
    return plot_data


def label_rate_bars(ax):
    for container in ax.containers:
        labels = [
            f"{bar.get_height():.1%}" if not np.isnan(bar.get_height()) else ""
            for bar in container
        ]
        ax.bar_label(container, labels=labels, padding=3, fontsize=9)


def label_count_bars(ax):
    for container in ax.containers:
        labels = [
            f"{int(round(bar.get_height()))}" if bar.get_height() > 0 else ""
            for bar in container
        ]
        ax.bar_label(container, labels=labels, padding=3, fontsize=9)

def plot_flip_analysis_by_error_type(
    model_outputs,
    title="Decision Flip Analysis by Error Type",
    figsize=(14, 6),
    output_path=None,
):
    required_keys = {"flip_error_summary"}
    missing_keys = sorted(required_keys - set(model_outputs))
    if missing_keys:
        raise ValueError(f"model_outputs is missing required keys: {missing_keys}")

    summary = model_outputs["flip_error_summary"].copy()
    direction_order = [
        "LR approve / RF reject",
        "RF approve / LR reject",
    ]
    error_type_order = [
        "Type I Error",
        "Type II Error",
    ]

    if summary.empty:
        fig, ax = plt.subplots(figsize=figsize)
        ax.text(0.5, 0.5, "No decision flips found.", ha="center", va="center", fontsize=12)
        ax.axis("off")
        if output_path is not None:
            fig.savefig(output_path, dpi=300, bbox_inches="tight")
        return summary, fig

    fig, axes = plt.subplots(1, 2, figsize=figsize, sharey=True)
    palette = {
        "Type I Error": "#DD8452",
        "Type II Error": "#4C78A8",
    }

    for ax, direction in zip(axes, direction_order):
        direction_data = summary.loc[summary["flip_direction"] == direction].copy()
        sns.barplot(
            data=direction_data,
            x="threshold_label",
            y="flip_count",
            hue="error_type",
            hue_order=error_type_order,
            palette=palette,
            ax=ax,
        )

        ax.set_title(direction, fontsize=12, fontweight="bold")
        ax.set_xlabel("Threshold")
        ax.set_ylabel("Flipped Observations" if direction == direction_order[0] else "")
        ax.tick_params(axis="x", rotation=0)
        label_count_bars(ax)

        legend = ax.get_legend()
        if direction == direction_order[0]:
            ax.legend(title="Error Type")
        elif legend is not None:
            legend.remove()

    fig.suptitle(
        title + "\nType I: approve return below threshold | Type II: reject return at or above threshold",
        fontsize=14,
        fontweight="bold",
        y=1.03,
    )
    fig.tight_layout()

    if output_path is not None:
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    return summary, fig


def add_threshold_percentage_labels(results, threshold_column="threshold"):
    results = results.copy()
    results["threshold_label"] = results[threshold_column].map(lambda x: f"{x * 100:g}%")
    return results


def get_threshold_label_order(results, threshold_column="threshold", label_column="threshold_label"):
    return (
        results[[threshold_column, label_column]]
        .drop_duplicates()
        .sort_values(threshold_column)[label_column]
        .tolist()
    )

def annotate_value_bars(ax, decimals=2):
    max_height = max(
        (bar.get_height() for container in ax.containers for bar in container),
        default=0,
    )
    min_height = min(
        (bar.get_height() for container in ax.containers for bar in container),
        default=0,
    )

    upper_limit = max_height * 1.15 if max_height > 0 else 1
    lower_limit = min(0, min_height * 1.15)
    ax.set_ylim(lower_limit, upper_limit)

    for container in ax.containers:
        labels = [
            f"{bar.get_height():,.{decimals}f}" if not np.isnan(bar.get_height()) else ""
            for bar in container
        ]
        ax.bar_label(container, labels=labels, padding=3, fontsize=10)


def annotate_rate_bars(ax):
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    max_height = max(
        (bar.get_height() for container in ax.containers for bar in container),
        default=0,
    )
    if max_height > 0:
        ax.set_ylim(0, max_height * 1.15)

    for container in ax.containers:
        labels = [
            f"{bar.get_height():.1%}" if not np.isnan(bar.get_height()) else ""
            for bar in container
        ]
        ax.bar_label(container, labels=labels, padding=3, fontsize=10)       

In [ ]:
def build_plot_base_frame(test_target, linear_predictions, random_forest_predictions):
    return pd.DataFrame(
        {
            "actual_return": pd.Series(test_target).reset_index(drop=True),
            "linear_prediction": pd.Series(linear_predictions).reset_index(drop=True),
            "random_forest_prediction": pd.Series(random_forest_predictions).reset_index(drop=True),
        }
    )


def build_threshold_frame(plot_base, threshold):
    threshold_frame = plot_base.copy()
    threshold_frame["threshold"] = threshold
    threshold_frame["LinearRegression_approve"] = threshold_frame["linear_prediction"] >= threshold
    threshold_frame["RandomForest_approve"] = threshold_frame["random_forest_prediction"] >= threshold
    threshold_frame["flip_direction"] = np.select(
        [
            threshold_frame["LinearRegression_approve"] & ~threshold_frame["RandomForest_approve"],
            ~threshold_frame["LinearRegression_approve"] & threshold_frame["RandomForest_approve"],
        ],
        [
            "LR approve / RF reject",
            "RF approve / LR reject",
        ],
        default="Agree",
    )
    return threshold_frame


def build_observation_level_frame(threshold_frame):
    frames = []

    for model_name, approve_column in [
        ("LinearRegression", "LinearRegression_approve"),
        ("RandomForest", "RandomForest_approve"),
    ]:
        model_frame = threshold_frame[["threshold", "actual_return", approve_column]].copy()
        model_frame["model"] = model_name
        model_frame["decision"] = np.where(model_frame[approve_column], "Approved", "Rejected")
        model_frame = model_frame.rename(columns={approve_column: "approve"})
        frames.append(model_frame[["threshold", "model", "decision", "approve", "actual_return"]])

    return pd.concat(frames, ignore_index=True)


def build_flip_level_frame(threshold_frame):
    flipped = threshold_frame.loc[threshold_frame["flip_direction"] != "Agree"].copy()

    if flipped.empty:
        return pd.DataFrame(columns=["threshold", "actual_return", "flip_direction"])

    return flipped[["threshold", "actual_return", "flip_direction"]]


def prepare_decision_plot_data(model_outputs):
    detailed_results = add_threshold_percentage_labels(model_outputs["detailed_results"].copy())
    flip_summary = add_threshold_percentage_labels(model_outputs["flip_summary_table"].copy())

    thresholds = sorted(detailed_results["threshold"].unique())
    plot_base = build_plot_base_frame(
        test_target=model_outputs["test_target"],
        linear_predictions=model_outputs["linear_predictions"],
        random_forest_predictions=model_outputs["random_forest_predictions"],
    )

    obs_frames = []
    flip_frames = []

    for threshold in thresholds:
        threshold_frame = build_threshold_frame(plot_base, threshold)
        obs_frames.append(build_observation_level_frame(threshold_frame))

        flip_frame = build_flip_level_frame(threshold_frame)
        if not flip_frame.empty:
            flip_frames.append(flip_frame)

    obs_level = add_threshold_percentage_labels(pd.concat(obs_frames, ignore_index=True))

    if flip_frames:
        flip_level = add_threshold_percentage_labels(pd.concat(flip_frames, ignore_index=True))
    else:
        flip_level = add_threshold_percentage_labels(
            pd.DataFrame(columns=["threshold", "actual_return", "flip_direction"])
        )

    return {
        "detailed_results": detailed_results,
        "flip_summary": flip_summary,
        "obs_level": obs_level,
        "flip_level": flip_level,
        "threshold_order": get_threshold_label_order(detailed_results),
    }


def plot_flip_direction_counts(plot_data):
    flip_summary = plot_data["flip_summary"]
    threshold_order = plot_data["threshold_order"]

    flip_dir_plot = flip_summary.melt(
        id_vars=["threshold", "threshold_label"],
        value_vars=["linear_approve_rf_reject", "rf_approve_linear_reject"],
        var_name="flip_direction",
        value_name="count",
    )
    flip_dir_plot["flip_direction"] = flip_dir_plot["flip_direction"].replace(
        {
            "linear_approve_rf_reject": "LR approve / RF reject",
            "rf_approve_linear_reject": "RF approve / LR reject",
        }
    )

    lr_counts = flip_dir_plot.loc[
        flip_dir_plot["flip_direction"] == "LR approve / RF reject",
        "count",
    ].to_numpy()
    rf_counts = flip_dir_plot.loc[
        flip_dir_plot["flip_direction"] == "RF approve / LR reject",
        "count",
    ].to_numpy()

    x = np.arange(len(threshold_order))

    plt.figure(figsize=(10, 6))
    plt.bar(x, lr_counts, label="LR approve / RF reject")
    plt.bar(x, rf_counts, bottom=lr_counts, label="RF approve / LR reject")
    plt.xticks(x, threshold_order)
    plt.title("Direction of Flips by Threshold")
    plt.xlabel("Threshold")
    plt.ylabel("Number of Flips")
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_approved_investments(plot_data):
    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=plot_data["detailed_results"],
        x="threshold_label",
        y="approved_count",
        hue="model",
        order=plot_data["threshold_order"],
    )
    plt.title("Approved Investments by Model and Threshold")
    plt.xlabel("Threshold")
    plt.ylabel("Approved Count")
    plt.tight_layout()
    plt.show()


def plot_approved_return_average(plot_data):
    plt.figure(figsize=(10, 6))
    sns.barplot(
        data=plot_data["detailed_results"],
        x="threshold_label",
        y="approved_return_avg",
        hue="model",
        order=plot_data["threshold_order"],
    )
    plt.title("Average Realized Return of Approved Investments")
    plt.xlabel("Threshold")
    plt.ylabel("Average Approved Return")
    plt.tight_layout()
    plt.show()


def plot_approved_return_distribution(plot_data):
    approved_only = plot_data["obs_level"].loc[plot_data["obs_level"]["decision"] == "Approved"].copy()

    plt.figure(figsize=(12, 6))
    sns.boxplot(
        data=approved_only,
        x="threshold_label",
        y="actual_return",
        hue="model",
        order=plot_data["threshold_order"],
    )
    plt.title("Return Distribution of Approved Investments")
    plt.xlabel("Threshold")
    plt.ylabel("Realized Return")
    plt.tight_layout()
    plt.show()


def plot_average_flip_return(plot_data):
    summary = (
        plot_data["flip_level"]
        .groupby(["threshold", "threshold_label"], as_index=False)
        .agg(avg_realized_return=("actual_return", "mean"))
        .sort_values("threshold")
    )

    plt.figure(figsize=(9, 5))
    sns.barplot(
        data=summary,
        x="threshold_label",
        y="avg_realized_return",
        color="steelblue",
        order=plot_data["threshold_order"],
    )
    plt.title("Average Realized Return for Flipped Decisions")
    plt.xlabel("Threshold")
    plt.ylabel("Average Realized Return")
    plt.tight_layout()
    plt.show()


def plot_approval_decline(plot_data):
    plt.figure(figsize=(10, 6))
    sns.lineplot(
        data=plot_data["detailed_results"],
        x="threshold_label",
        y="approved_count",
        hue="model",
        marker="o",
        linewidth=2.5,
        sort=False,
    )
    plt.title("Approved Investments Decline as Threshold Increases")
    plt.xlabel("Threshold")
    plt.ylabel("Approved Count")
    plt.tight_layout()
    plt.show()
